# Inference-only notebook per migliorare il RE BERT

Obiettivo: migliorare le metriche del modello già allenato in `src/re/models/bert_biomedbert_re` **senza retraining**.

In questo notebook facciamo solo ottimizzazione della fase di inferenza:

1. carichiamo il modello già allenato
2. facciamo una sola forward pass sul dev
3. salviamo i **logits completi** per ogni coppia candidata
4. proviamo decoder più intelligenti:
   - **legal-only decoding**
   - **temperature scaling**
   - **global thresholds tuning**
   - **predicate-specific thresholds**
5. confrontiamo tutto con micro/macro F1 sul dev

## Cosa stiamo migliorando davvero rispetto al notebook bert_RE_margin

Le novità più utili qui sono:

1. **best legal predicate invece di argmax globale**
   - prima, se il top label era illegale, perdevi la seconda miglior label legale
   - ora scegli direttamente tra i predicati ammessi per quel pair

2. **temperature scaling**
   - aiuta a calibrare le probabilità in inference

3. **full logits cache**
   - consente tanti esperimenti senza rilanciare il modello

4. **threshold multipli per predicato**
   - non solo `margin`, ma anche `min_prob` e `max_pair_chars`

Questa è probabilmente la direzione migliore per spremere ancora un po’ di F1 senza toccare il training.

### Config

In [1]:
from pathlib import Path
import os
import json
import math
import random
import itertools
import re
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel

MODEL_DIR = Path("models/bert_biomedbert_re").resolve()
DEV_PATH = Path("../../data/GutBrainIE_Full_Collection_2026/Annotations/Dev/json_format/dev.json")

BATCH_SIZE = 16
MAX_LENGTH = 512
WINDOW_CHARS = 300

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("MODEL_DIR =", MODEL_DIR)
print("DEV_PATH =", DEV_PATH)
print("DEVICE =", DEVICE)
DEVICE

C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MODEL_DIR = C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\re\models\bert_biomedbert_re
DEV_PATH = ..\..\data\GutBrainIE_Full_Collection_2026\Annotations\Dev\json_format\dev.json
DEVICE = cuda


device(type='cuda')

## Check local model directory

In [2]:
print("exists =", MODEL_DIR.exists())
print("is_dir =", MODEL_DIR.is_dir())

if MODEL_DIR.exists():
    print("\nFiles:")
    for p in sorted(MODEL_DIR.iterdir()):
        print(" -", p.name)
else:
    raise FileNotFoundError(f"Local model directory not found: {MODEL_DIR}")

exists = True
is_dir = True

Files:
 - cache_tok
 - checkpoint-30135
 - checkpoint-5000
 - label_mappings.json
 - pytorch_model.bin
 - tokenizer.json
 - tokenizer_config.json
 - training_args.bin


## Helper to find last checkpoint

In [3]:
def find_last_checkpoint_local(root_dir: Path) -> Path:
    ckpts = [d for d in root_dir.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")]
    if not ckpts:
        return root_dir
    ckpts = sorted(ckpts, key=lambda x: int(x.name.split("-")[1]))
    return ckpts[-1]

LOAD_DIR = find_last_checkpoint_local(MODEL_DIR)
STATE_PATH = LOAD_DIR / "pytorch_model.bin"

print("LOAD_DIR =", LOAD_DIR)
print("STATE_PATH =", STATE_PATH)

if not STATE_PATH.exists():
    raise FileNotFoundError(f"Checkpoint file not found: {STATE_PATH}")

LOAD_DIR = C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\re\models\bert_biomedbert_re\checkpoint-30135
STATE_PATH = C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\re\models\bert_biomedbert_re\checkpoint-30135\pytorch_model.bin


In [4]:
LEGAL_ENTITY_LABELS = {
    "anatomical location","animal","bacteria","biomedical technique","chemical","DDF",
    "dietary supplement","drug","food","gene","human","microbiome","statistical technique"
}
LEGAL_RELATION_LABELS = {
    "administered","affect","change abundance","change effect","change expression","compared to",
    "impact","influence","interact","is a","is linked to","located in","part of","produced by",
    "strike","target","used by"
}

RELATION_LABELS = [
    "no relation",
    "administered",
    "affect",
    "change abundance",
    "change effect",
    "change expression",
    "compared to",
    "impact",
    "influence",
    "interact",
    "is a",
    "is linked to",
    "located in",
    "part of",
    "produced by",
    "strike",
    "target",
    "used by"
]

label2id = {label: idx for idx, label in enumerate(RELATION_LABELS)}
id2label = {idx: label for idx, label in enumerate(RELATION_LABELS)}

def norm_ent(label: str) -> str:
    if label is None:
        return ""
    lab = str(label).strip()
    if lab.lower() == "ddf":
        return "DDF"
    return lab

def norm_span(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

LEGAL_RELATIONS = [
    ("DDF", "affect", "DDF"),
    ("microbiome", "is linked to", "DDF"),
    ("DDF", "target", "human"),
    ("drug", "change effect", "DDF"),
    ("DDF", "is a", "DDF"),
    ("microbiome", "located in", "human"),
    ("chemical", "influence", "DDF"),
    ("dietary supplement", "influence", "DDF"),
    ("DDF", "target", "animal"),
    ("chemical", "impact", "microbiome"),
    ("anatomical location", "located in", "animal"),
    ("microbiome", "located in", "animal"),
    ("chemical", "located in", "anatomical location"),
    ("bacteria", "part of", "microbiome"),
    ("DDF", "strike", "anatomical location"),
    ("drug", "administered", "animal"),
    ("bacteria", "influence", "DDF"),
    ("drug", "impact", "microbiome"),
    ("DDF", "change abundance", "microbiome"),
    ("microbiome", "located in", "anatomical location"),
    ("microbiome", "used by", "biomedical technique"),
    ("chemical", "produced by", "microbiome"),
    ("dietary supplement", "impact", "microbiome"),
    ("bacteria", "located in", "animal"),
    ("animal", "used by", "biomedical technique"),
    ("chemical", "impact", "bacteria"),
    ("chemical", "located in", "animal"),
    ("food", "impact", "bacteria"),
    ("microbiome", "compared to", "microbiome"),
    ("human", "used by", "biomedical technique"),
    ("bacteria", "change expression", "gene"),
    ("chemical", "located in", "human"),
    ("drug", "interact", "chemical"),
    ("food", "administered", "human"),
    ("DDF", "change abundance", "bacteria"),
    ("chemical", "interact", "chemical"),
    ("chemical", "part of", "chemical"),
    ("dietary supplement", "impact", "bacteria"),
    ("DDF", "interact", "chemical"),
    ("food", "impact", "microbiome"),
    ("food", "influence", "DDF"),
    ("bacteria", "located in", "human"),
    ("dietary supplement", "administered", "human"),
    ("bacteria", "interact", "chemical"),
    ("drug", "change expression", "gene"),
    ("drug", "impact", "bacteria"),
    ("drug", "administered", "human"),
    ("anatomical location", "located in", "human"),
    ("dietary supplement", "change expression", "gene"),
    ("chemical", "change expression", "gene"),
    ("bacteria", "interact", "bacteria"),
    ("drug", "interact", "drug"),
    ("microbiome", "change expression", "gene"),
    ("bacteria", "interact", "drug"),
    ("food", "change expression", "gene")
]

legal_pairs = {}
for s, p, o in LEGAL_RELATIONS:
    s = norm_ent(s)
    o = norm_ent(o)
    legal_pairs.setdefault((s, o), set()).add(p)

print("num relation labels:", len(RELATION_LABELS))
print("num legal type pairs:", len(legal_pairs))

num relation labels: 18
num legal type pairs: 52


### Utility functions : text, offsets and metrics

In [5]:
def load_re_data(file_paths):
    all_data = {}
    for file_path in file_paths:
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            all_data.update(data)
            print(f"Loaded {len(data)} docs from {os.path.basename(file_path)}")
    return all_data

def create_full_text_with_offsets(title, abstract):
    full_text = f"{title} {abstract}"
    abstract_offset = len(title) + 1
    return full_text, abstract_offset

def adjust_entity_positions(entity, abstract_offset):
    if entity['location'] == 'abstract':
        return {
            'start_idx': entity['start_idx'] + abstract_offset,
            'end_idx': entity['end_idx'] + abstract_offset,
            'text_span': entity['text_span'],
            'label': entity['label']
        }
    else:
        return {
            'start_idx': entity['start_idx'],
            'end_idx': entity['end_idx'],
            'text_span': entity['text_span'],
            'label': entity['label']
        }

def insert_entity_markers(text, subject, obj):
    entities = [
        (subject['start_idx'], subject['end_idx'], '[E1]', '[/E1]'),
        (obj['start_idx'], obj['end_idx'], '[E2]', '[/E2]')
    ]
    entities = sorted(entities, key=lambda x: x[0])

    marked_text = text
    offset = 0
    for start, end, start_marker, end_marker in entities:
        adj_start = start + offset
        adj_end = end + offset + 1
        marked_text = (
            marked_text[:adj_start]
            + start_marker
            + marked_text[adj_start:adj_end]
            + end_marker
            + marked_text[adj_end:]
        )
        offset += len(start_marker) + len(end_marker)
    return marked_text

def build_window_around_entities(text, subject, obj, window_chars=300):
    s_start, s_end = subject["start_idx"], subject["end_idx"]
    o_start, o_end = obj["start_idx"], obj["end_idx"]

    left = min(s_start, o_start)
    right = max(s_end, o_end)

    win_start = max(0, left - window_chars)
    win_end = min(len(text) - 1, right + window_chars)

    window_text = text[win_start:win_end + 1]

    subj_w = dict(subject)
    obj_w = dict(obj)

    subj_w["start_idx"] = s_start - win_start
    subj_w["end_idx"] = s_end - win_start
    obj_w["start_idx"] = o_start - win_start
    obj_w["end_idx"] = o_end - win_start

    return window_text, subj_w, obj_w

def build_marked_text_for_inference(text, subj, obj, window_chars=300):
    w_text, w_subj, w_obj = build_window_around_entities(text, subj, obj, window_chars=window_chars)
    return insert_entity_markers(w_text, w_subj, w_obj)

def gold_tuple(r):
    return (
        norm_span(r["subject_text_span"]),
        norm_ent(r["subject_label"]),
        r["predicate"].strip(),
        norm_span(r["object_text_span"]),
        norm_ent(r["object_label"]),
    )

def build_gold_maps(dev_data):
    gold_by_doc = {}
    for pmid, art in dev_data.items():
        s = set()
        for r in art.get("mention_level_relations", []):
            pred = r["predicate"].strip()
            if pred not in LEGAL_RELATION_LABELS:
                continue
            s.add(gold_tuple(r))
        gold_by_doc[str(pmid)] = s
    return gold_by_doc

def micro_scores(gold_by_doc, pred_by_doc):
    tp = fp = fn = 0
    for pmid, g in gold_by_doc.items():
        p = pred_by_doc.get(pmid, set())
        tp += len(g & p)
        fp += len(p - g)
        fn += len(g - p)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) else 0.0
    return {"P": prec, "R": rec, "F1": f1, "TP": tp, "FP": fp, "FN": fn}

def macro_scores(gold_by_doc, pred_by_doc):
    preds = sorted({t[2] for s in gold_by_doc.values() for t in s} | {t[2] for s in pred_by_doc.values() for t in s})
    vals = []
    for pred in preds:
        tp = fp = fn = 0
        for pmid, g in gold_by_doc.items():
            g_pred = {t for t in g if t[2] == pred}
            p_pred = {t for t in pred_by_doc.get(pmid, set()) if t[2] == pred}
            tp += len(g_pred & p_pred)
            fp += len(p_pred - g_pred)
            fn += len(g_pred - p_pred)
        P = tp / (tp + fp) if (tp + fp) else 0.0
        R = tp / (tp + fn) if (tp + fn) else 0.0
        F1 = (2 * P * R) / (P + R) if (P + R) else 0.0
        vals.append((P, R, F1))
    return {
        "macro_P": float(np.mean([x[0] for x in vals])) if vals else 0.0,
        "macro_R": float(np.mean([x[1] for x in vals])) if vals else 0.0,
        "macro_F1": float(np.mean([x[2] for x in vals])) if vals else 0.0,
    }

## Load Dev Set

In [6]:
dev_data = load_re_data([DEV_PATH])
gold_by_doc = build_gold_maps(dev_data)

print("dev docs:", len(dev_data))
print("gold relations:", sum(len(v) for v in gold_by_doc.values()))

Loaded 80 docs from dev.json
dev docs: 80
gold relations: 1139


##  Load trained model and tokenizer

In [7]:
class BertForREWithEntityMarkers(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        hidden_size = self.bert.config.hidden_size
        self.classifier = nn.Linear(hidden_size * 2, num_labels)
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask, e1_mask, e2_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state

        e1_h = torch.bmm(e1_mask.unsqueeze(1).float(), sequence_output).squeeze(1)
        e2_h = torch.bmm(e2_mask.unsqueeze(1).float(), sequence_output).squeeze(1)

        concat_h = torch.cat([e1_h, e2_h], dim=-1)
        concat_h = self.dropout(concat_h)
        logits = self.classifier(concat_h)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits.view(-1, self.num_labels), labels.view(-1))

        return {"loss": loss, "logits": logits}


## Load tokenizer from local directory

In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    use_fast=True,
    local_files_only=True,
)

e1_token_id = tokenizer.convert_tokens_to_ids("[E1]")
e2_token_id = tokenizer.convert_tokens_to_ids("[E2]")

print("Tokenizer loaded")
print("[E1] id:", e1_token_id)
print("[E2] id:", e2_token_id)

Tokenizer loaded
[E1] id: 30522
[E2] id: 30524


## Load trained model

In [9]:
BASE_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

model = BertForREWithEntityMarkers(
    model_name=BASE_MODEL_NAME,
    num_labels=len(RELATION_LABELS)
)

model.bert.resize_token_embeddings(len(tokenizer))

state_dict = torch.load(STATE_PATH, map_location="cpu")
model.load_state_dict(state_dict)

model.to(DEVICE)
model.eval()

print("Model loaded successfully on", DEVICE)

C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\super\.cache\huggingface\hub\models--microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 

Model loaded successfully on cuda


## Cache
Qui invece cacheiamo **tutto il vettore dei logits** per ogni coppia.
Questo permette decoder più forti:

1. **best legal predicate per type-pair**
2. **temperature scaling**
3. **fallback sulla seconda miglior label legale**
4. threshold diversi per predicato

In [10]:
@torch.no_grad()
def cache_dev_logits_once(
    model,
    tokenizer,
    dev_data,
    legal_pairs,
    e1_token_id,
    e2_token_id,
    device,
    batch_size=16,
    max_length=512,
    window_chars=300,
    cache_path=None,
):
    if cache_path and os.path.exists(cache_path):
        print("[cache] loading:", cache_path)
        return torch.load(cache_path, map_location="cpu")

    model.eval()
    cache = {}

    for pmid, article in tqdm(dev_data.items(), desc="Caching full logits on dev"):
        title = article["metadata"]["title"]
        abstract = article["metadata"]["abstract"]
        full_text, abstract_offset = create_full_text_with_offsets(title, abstract)

        adjusted_entities = [
            {
                **adjust_entity_positions(e, abstract_offset),
                "label": norm_ent(e["label"]),
                "text_span": norm_span(e["text_span"]),
            }
            for e in article["entities"]
        ]

        pair_examples = []
        rows_meta = []

        for i, subj in enumerate(adjusted_entities):
            for j, obj in enumerate(adjusted_entities):
                if i == j:
                    continue

                type_pair = (subj["label"], obj["label"])
                if type_pair not in legal_pairs:
                    continue

                pair_examples.append((full_text, subj, obj))
                rows_meta.append({
                    "k": (subj["text_span"], subj["label"], obj["text_span"], obj["label"]),
                    "dist": abs(subj["start_idx"] - obj["start_idx"]),
                    "subject_label": subj["label"],
                    "object_label": obj["label"],
                })

        if not pair_examples:
            cache[str(pmid)] = []
            continue

        marked_texts = [
            build_marked_text_for_inference(text, subj, obj, window_chars=window_chars)
            for (text, subj, obj) in pair_examples
        ]

        rows = []
        idx = 0

        for start in range(0, len(marked_texts), batch_size):
            batch_texts = marked_texts[start:start+batch_size]
            enc = tokenizer(
                batch_texts,
                truncation=True,
                max_length=max_length,
                padding=True,
                return_tensors="pt",
            )

            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)
            e1_mask = (input_ids == e1_token_id).long()
            e2_mask = (input_ids == e2_token_id).long()

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                e1_mask=e1_mask,
                e2_mask=e2_mask
            )["logits"]

            logits = logits.detach().cpu().float()

            for b in range(logits.shape[0]):
                row = dict(rows_meta[idx])
                row["logits"] = logits[b]
                rows.append(row)
                idx += 1

        cache[str(pmid)] = rows

    if cache_path:
        torch.save(cache, cache_path)
        print("[cache] saved:", cache_path)

    return cache

In [11]:
FULL_CACHE_PATH = os.path.join(MODEL_DIR, "dev_logits_cache.pt")

dev_cache = cache_dev_logits_once(
    model=model,
    tokenizer=tokenizer,
    dev_data=dev_data,
    legal_pairs=legal_pairs,
    e1_token_id=e1_token_id,
    e2_token_id=e2_token_id,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    window_chars=WINDOW_CHARS,
    cache_path=FULL_CACHE_PATH,
)

print("cached docs:", len(dev_cache))
print("total cached pairs:", sum(len(v) for v in dev_cache.values()))

Caching full logits on dev: 100%|██████████| 80/80 [05:42<00:00,  4.28s/it]


[cache] saved: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\re\models\bert_biomedbert_re\dev_logits_cache.pt
cached docs: 80
total cached pairs: 50202


## Decoder 1: legal-only decoding con temperature scaling

Idea chiave:

per una coppia `(subject_label, object_label)` non ha senso scegliere il top label su **tutte** le classi.

Meglio fare decoding solo su:
- `"no relation"`
- i predicati legali per quel type-pair

In più introduciamo una temperatura `T`:

- `T < 1`: distribuzione più sharp
- `T > 1`: distribuzione più morbida

Questo è spesso meglio del decoder vecchio perché evita di scartare coppie dove il top globale era illegale ma il **top legale** era corretto.

In [12]:
def softmax_np(x):
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / ex.sum()

def row_to_legal_scores(row, legal_pairs, label2id, id2label, temperature=1.0, renorm_legal=True):
    """
    Restituisce:
      pred_label, p_best, p_no, margin_val
    dove pred_label è la miglior relazione legale per quel type-pair.
    """
    s_lab = row["subject_label"]
    o_lab = row["object_label"]
    allowed_preds = sorted(list(legal_pairs.get((s_lab, o_lab), [])))

    if not allowed_preds:
        return None

    logits = row["logits"].numpy() / temperature

    if renorm_legal:
        allowed_ids = [label2id["no relation"]] + [label2id[p] for p in allowed_preds]
        probs_sub = softmax_np(logits[allowed_ids])

        p_no = float(probs_sub[0])
        best_idx = int(np.argmax(probs_sub[1:])) + 1
        pred_label = id2label[allowed_ids[best_idx]]
        p_best = float(probs_sub[best_idx])
    else:
        probs = softmax_np(logits)
        p_no = float(probs[label2id["no relation"]])

        legal_ids = [label2id[p] for p in allowed_preds]
        best_legal_id = max(legal_ids, key=lambda idx: probs[idx])
        pred_label = id2label[best_legal_id]
        p_best = float(probs[best_legal_id])

    return {
        "pred_label": pred_label,
        "p_best": p_best,
        "p_no": p_no,
        "margin_val": p_best - p_no
    }

## Decoder 2: decoding globale con parametri tunabili

Qui costruiamo un decoder più ricco del precedente.

Per ogni coppia:
- applichiamo `dist <= max_pair_chars`
- scegliamo il best legal predicate
- teniamo la coppia solo se:
  - `p_best >= min_prob`
  - `p_best - p_no >= margin`

Poi deduplichiamo mantenendo il miglior score per ogni pair.

In [13]:
def decode_doc_global(
    rows,
    legal_pairs,
    label2id,
    id2label,
    max_pair_chars=300,
    min_prob=0.25,
    margin=0.20,
    temperature=1.0,
    renorm_legal=True,
):
    best_for_pair = {}

    for row in rows:
        if row["dist"] > max_pair_chars:
            continue

        scores = row_to_legal_scores(
            row,
            legal_pairs=legal_pairs,
            label2id=label2id,
            id2label=id2label,
            temperature=temperature,
            renorm_legal=renorm_legal,
        )
        if scores is None:
            continue

        if scores["p_best"] < min_prob:
            continue
        if scores["margin_val"] < margin:
            continue

        k = row["k"]
        prev = best_for_pair.get(k)
        if prev is None or scores["p_best"] > prev[1]:
            best_for_pair[k] = (scores["pred_label"], scores["p_best"])

    out_set = set()
    for (s_text, s_lab, o_text, o_lab), (pred, score) in best_for_pair.items():
        out_set.add((s_text, s_lab, pred, o_text, o_lab))

    return out_set, best_for_pair

def build_predictions_json_from_best(best_for_pair):
    mention_level = []
    for (s_text, s_lab, o_text, o_lab), (pred, score) in sorted(best_for_pair.items()):
        mention_level.append({
            "subject_text_span": s_text,
            "subject_label": s_lab,
            "predicate": pred,
            "object_text_span": o_text,
            "object_label": o_lab
        })
    return mention_level

## Baseline inference attuale da confrontare

Prima di migliorare, misuriamo la baseline attuale con una configurazione simile alla tua migliore globale:

- `MAX_PAIR_CHARS = 300`
- `MIN_PROB = 0.25`
- `MARGIN = 0.20`

ma già con **legal-only decoding**, che è la prima modifica che vogliamo testare.

In [14]:
pred_by_doc_base = {}
predictions_base = {}

for pmid, rows in dev_cache.items():
    pred_set, best_for_pair = decode_doc_global(
        rows,
        legal_pairs=legal_pairs,
        label2id=label2id,
        id2label=id2label,
        max_pair_chars=300,
        min_prob=0.25,
        margin=0.20,
        temperature=1.0,
        renorm_legal=True,
    )
    pred_by_doc_base[pmid] = pred_set
    predictions_base[pmid] = {"mention_level_relations": build_predictions_json_from_best(best_for_pair)}

micro_base = micro_scores(gold_by_doc, pred_by_doc_base)
macro_base = macro_scores(gold_by_doc, pred_by_doc_base)

print("=== Baseline legal-only ===")
print(micro_base)
print(macro_base)

=== Baseline legal-only ===
{'P': 0.48172323759791125, 'R': 0.6479367866549605, 'F1': 0.5526020217147136, 'TP': 738, 'FP': 794, 'FN': 401}
{'macro_P': 0.4583215878976895, 'macro_R': 0.6128277250936436, 'macro_F1': 0.4929603270235719}


## Grid search inference-only

Adesso facciamo tuning vero del decoder sui parametri:

- `max_pair_chars`
- `min_prob`
- `margin`
- `temperature`
- `renorm_legal`

Questo è il cuore del notebook: nessun retraining, solo miglior decoding.

In [15]:
MAX_PAIR_CHARS_GRID = [250, 300, 350, 400, 500]
MIN_PROB_GRID = [0.10, 0.20, 0.25, 0.30, 0.35]
MARGIN_GRID = [0.05, 0.10, 0.15, 0.20, 0.25]
TEMP_GRID = [0.75, 0.90, 1.00, 1.10, 1.25]
RENORM_GRID = [True, False]

grid_results = []

for max_chars, min_prob, margin, temp, renorm in tqdm(
    list(itertools.product(MAX_PAIR_CHARS_GRID, MIN_PROB_GRID, MARGIN_GRID, TEMP_GRID, RENORM_GRID)),
    desc="Global grid search"
):
    pred_by_doc = {}
    for pmid, rows in dev_cache.items():
        pred_set, _ = decode_doc_global(
            rows,
            legal_pairs=legal_pairs,
            label2id=label2id,
            id2label=id2label,
            max_pair_chars=max_chars,
            min_prob=min_prob,
            margin=margin,
            temperature=temp,
            renorm_legal=renorm,
        )
        pred_by_doc[pmid] = pred_set

    micro = micro_scores(gold_by_doc, pred_by_doc)
    macro = macro_scores(gold_by_doc, pred_by_doc)

    grid_results.append({
        "MAX_PAIR_CHARS": max_chars,
        "MIN_PROB": min_prob,
        "MARGIN": margin,
        "TEMP": temp,
        "RENORM_LEGAL": renorm,
        **micro,
        **macro,
    })

grid_results = sorted(grid_results, key=lambda x: x["F1"], reverse=True)

print("Top 10 global configs:")
for r in grid_results[:10]:
    print(r)

BEST_GLOBAL = grid_results[0]
BEST_GLOBAL

Global grid search: 100%|██████████| 1250/1250 [08:55<00:00,  2.33it/s]

Top 10 global configs:
{'MAX_PAIR_CHARS': 250, 'MIN_PROB': 0.1, 'MARGIN': 0.25, 'TEMP': 1.25, 'RENORM_LEGAL': True, 'P': 0.49793672627235214, 'R': 0.6356453028972783, 'F1': 0.5584265329733898, 'TP': 724, 'FP': 730, 'FN': 415, 'macro_P': 0.47448121377417574, 'macro_R': 0.5934317700932226, 'macro_F1': 0.49312545875448127}
{'MAX_PAIR_CHARS': 250, 'MIN_PROB': 0.2, 'MARGIN': 0.25, 'TEMP': 1.25, 'RENORM_LEGAL': True, 'P': 0.49793672627235214, 'R': 0.6356453028972783, 'F1': 0.5584265329733898, 'TP': 724, 'FP': 730, 'FN': 415, 'macro_P': 0.47448121377417574, 'macro_R': 0.5934317700932226, 'macro_F1': 0.49312545875448127}
{'MAX_PAIR_CHARS': 250, 'MIN_PROB': 0.25, 'MARGIN': 0.25, 'TEMP': 1.25, 'RENORM_LEGAL': True, 'P': 0.49793672627235214, 'R': 0.6356453028972783, 'F1': 0.5584265329733898, 'TP': 724, 'FP': 730, 'FN': 415, 'macro_P': 0.47448121377417574, 'macro_R': 0.5934317700932226, 'macro_F1': 0.49312545875448127}
{'MAX_PAIR_CHARS': 250, 'MIN_PROB': 0.3, 'MARGIN': 0.25, 'TEMP': 1.25, 'RENORM_

{'MAX_PAIR_CHARS': 250,
 'MIN_PROB': 0.1,
 'MARGIN': 0.25,
 'TEMP': 1.25,
 'RENORM_LEGAL': True,
 'P': 0.49793672627235214,
 'R': 0.6356453028972783,
 'F1': 0.5584265329733898,
 'TP': 724,
 'FP': 730,
 'FN': 415,
 'macro_P': 0.47448121377417574,
 'macro_R': 0.5934317700932226,
 'macro_F1': 0.49312545875448127}

## Decode con la miglior configurazione globale trovata

Questa configurazione sarà il nostro nuovo fallback robusto, analogo al tuo `grid_best`, ma con decoder più forte.

In [16]:
BEST_MAX = BEST_GLOBAL["MAX_PAIR_CHARS"]
BEST_MINP = BEST_GLOBAL["MIN_PROB"]
BEST_MARG = BEST_GLOBAL["MARGIN"]
BEST_TEMP = BEST_GLOBAL["TEMP"]
BEST_RENORM = BEST_GLOBAL["RENORM_LEGAL"]

pred_by_doc_best = {}
predictions_best = {}

for pmid, rows in dev_cache.items():
    pred_set, best_for_pair = decode_doc_global(
        rows,
        legal_pairs=legal_pairs,
        label2id=label2id,
        id2label=id2label,
        max_pair_chars=BEST_MAX,
        min_prob=BEST_MINP,
        margin=BEST_MARG,
        temperature=BEST_TEMP,
        renorm_legal=BEST_RENORM,
    )
    pred_by_doc_best[pmid] = pred_set
    predictions_best[pmid] = {"mention_level_relations": build_predictions_json_from_best(best_for_pair)}

print("=== BEST GLOBAL ===")
print(micro_scores(gold_by_doc, pred_by_doc_best))
print(macro_scores(gold_by_doc, pred_by_doc_best))

=== BEST GLOBAL ===
{'P': 0.49793672627235214, 'R': 0.6356453028972783, 'F1': 0.5584265329733898, 'TP': 724, 'FP': 730, 'FN': 415}
{'macro_P': 0.47448121377417574, 'macro_R': 0.5934317700932226, 'macro_F1': 0.49312545875448127}


## Miglioramento 2: threshold specifici per predicato

Adesso facciamo un passo oltre:

per ogni predicato, invece di usare gli stessi valori globali per tutti, cerchiamo:

- `margin` per predicato
- `min_prob` per predicato
- opzionalmente `max_pair_chars` per predicato

Partiamo dalla miglior configurazione globale e la rifiniamo classe per classe.

In [17]:
PREDICATES_IN_DEV = sorted({t[2] for s in gold_by_doc.values() for t in s})

PRED_MARGIN_GRID = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
PRED_MINPROB_GRID = [0.00, 0.10, 0.20, 0.25, 0.30, 0.35]
PRED_MAXCHARS_GRID = [250, 300, 350, 400, 500]

In [18]:
def decode_doc_pred_specific(
    rows,
    legal_pairs,
    label2id,
    id2label,
    default_max_chars,
    default_min_prob,
    default_margin,
    temperature,
    renorm_legal,
    max_chars_by_pred=None,
    min_prob_by_pred=None,
    margin_by_pred=None,
):
    best_for_pair = {}

    for row in rows:
        scores = row_to_legal_scores(
            row,
            legal_pairs=legal_pairs,
            label2id=label2id,
            id2label=id2label,
            temperature=temperature,
            renorm_legal=renorm_legal,
        )
        if scores is None:
            continue

        pred = scores["pred_label"]

        maxc = max_chars_by_pred.get(pred, default_max_chars) if max_chars_by_pred else default_max_chars
        minp = min_prob_by_pred.get(pred, default_min_prob) if min_prob_by_pred else default_min_prob
        marg = margin_by_pred.get(pred, default_margin) if margin_by_pred else default_margin

        if row["dist"] > maxc:
            continue
        if scores["p_best"] < minp:
            continue
        if scores["margin_val"] < marg:
            continue

        k = row["k"]
        prev = best_for_pair.get(k)
        if prev is None or scores["p_best"] > prev[1]:
            best_for_pair[k] = (pred, scores["p_best"])

    out = set()
    for (s_text, s_lab, o_text, o_lab), (pred, score) in best_for_pair.items():
        out.add((s_text, s_lab, pred, o_text, o_lab))

    return out, best_for_pair

## Tuning greedy dei parametri per predicato

Per tenere il notebook gestibile, facciamo una strategia semplice:

1. partiamo dalla best global
2. per ogni predicato ottimizziamo separatamente:
   - margin
   - min_prob
   - max_pair_chars
3. poi ricombiniamo tutto nel decoder finale

Non è perfetto come una grid search totale, ma è molto più veloce e spesso funziona bene.

In [19]:
best_margin_by_pred = {}
best_minprob_by_pred = {}
best_maxchars_by_pred = {}

# STEP 1: margin per predicato
for pred in tqdm(PREDICATES_IN_DEV, desc="Tune margin by predicate"):
    best = None
    for m in PRED_MARGIN_GRID:
        pred_by_doc = {}
        for pmid, rows in dev_cache.items():
            pred_set, _ = decode_doc_pred_specific(
                rows=rows,
                legal_pairs=legal_pairs,
                label2id=label2id,
                id2label=id2label,
                default_max_chars=BEST_MAX,
                default_min_prob=BEST_MINP,
                default_margin=BEST_MARG,
                temperature=BEST_TEMP,
                renorm_legal=BEST_RENORM,
                margin_by_pred={pred: m},
            )
            pred_by_doc[pmid] = {t for t in pred_set if t[2] == pred}

        gold_pred = {pmid: {t for t in gold_by_doc[pmid] if t[2] == pred} for pmid in gold_by_doc}
        sc = micro_scores(gold_pred, pred_by_doc)

        if best is None or sc["F1"] > best["F1"]:
            best = {"margin": m, **sc}

    best_margin_by_pred[pred] = best["margin"]

best_margin_by_pred

Tune margin by predicate: 100%|██████████| 17/17 [01:57<00:00,  6.92s/it]


{'administered': 0.1,
 'affect': 0.3,
 'change abundance': 0.3,
 'change effect': 0.05,
 'change expression': 0.05,
 'compared to': 0.05,
 'impact': 0.3,
 'influence': 0.3,
 'interact': 0.3,
 'is a': 0.25,
 'is linked to': 0.1,
 'located in': 0.3,
 'part of': 0.15,
 'produced by': 0.15,
 'strike': 0.2,
 'target': 0.15,
 'used by': 0.2}

In [ ]:
# STEP 2: min_prob per predicato
for pred in tqdm(PREDICATES_IN_DEV, desc="Tune min_prob by predicate"):
    best = None
    for mp in PRED_MINPROB_GRID:
        pred_by_doc = {}
        for pmid, rows in dev_cache.items():
            pred_set, _ = decode_doc_pred_specific(
                rows=rows,
                legal_pairs=legal_pairs,
                label2id=label2id,
                id2label=id2label,
                default_max_chars=BEST_MAX,
                default_min_prob=BEST_MINP,
                default_margin=BEST_MARG,
                temperature=BEST_TEMP,
                renorm_legal=BEST_RENORM,
                margin_by_pred=best_margin_by_pred,
                min_prob_by_pred={pred: mp},
            )
            pred_by_doc[pmid] = {t for t in pred_set if t[2] == pred}

        gold_pred = {pmid: {t for t in gold_by_doc[pmid] if t[2] == pred} for pmid in gold_by_doc}
        sc = micro_scores(gold_pred, pred_by_doc)

        if best is None or sc["F1"] > best["F1"]:
            best = {"min_prob": mp, **sc}

    best_minprob_by_pred[pred] = best["min_prob"]

best_minprob_by_pred

In [ ]:
# STEP 3: max_pair_chars per predicato
for pred in tqdm(PREDICATES_IN_DEV, desc="Tune max_chars by predicate"):
    best = None
    for mc in PRED_MAXCHARS_GRID:
        pred_by_doc = {}
        for pmid, rows in dev_cache.items():
            pred_set, _ = decode_doc_pred_specific(
                rows=rows,
                legal_pairs=legal_pairs,
                label2id=label2id,
                id2label=id2label,
                default_max_chars=BEST_MAX,
                default_min_prob=BEST_MINP,
                default_margin=BEST_MARG,
                temperature=BEST_TEMP,
                renorm_legal=BEST_RENORM,
                margin_by_pred=best_margin_by_pred,
                min_prob_by_pred=best_minprob_by_pred,
                max_chars_by_pred={pred: mc},
            )
            pred_by_doc[pmid] = {t for t in pred_set if t[2] == pred}

        gold_pred = {pmid: {t for t in gold_by_doc[pmid] if t[2] == pred} for pmid in gold_by_doc}
        sc = micro_scores(gold_pred, pred_by_doc)

        if best is None or sc["F1"] > best["F1"]:
            best = {"max_chars": mc, **sc}

    best_maxchars_by_pred[pred] = best["max_chars"]

best_maxchars_by_pred

## Decode finale con parametri specifici per predicato

In [20]:
pred_by_doc_ps = {}
predictions_ps = {}

for pmid, rows in dev_cache.items():
    pred_set, best_for_pair = decode_doc_pred_specific(
        rows=rows,
        legal_pairs=legal_pairs,
        label2id=label2id,
        id2label=id2label,
        default_max_chars=BEST_MAX,
        default_min_prob=BEST_MINP,
        default_margin=BEST_MARG,
        temperature=BEST_TEMP,
        renorm_legal=BEST_RENORM,
        max_chars_by_pred=best_maxchars_by_pred,
        min_prob_by_pred=best_minprob_by_pred,
        margin_by_pred=best_margin_by_pred,
    )
    pred_by_doc_ps[pmid] = pred_set
    predictions_ps[pmid] = {"mention_level_relations": build_predictions_json_from_best(best_for_pair)}

print("=== PREDICATE-SPECIFIC ===")
print(micro_scores(gold_by_doc, pred_by_doc_ps))
print(macro_scores(gold_by_doc, pred_by_doc_ps))

=== PREDICATE-SPECIFIC ===
{'P': 0.5048076923076923, 'R': 0.6453028972783144, 'F1': 0.5664739884393064, 'TP': 735, 'FP': 721, 'FN': 404}
{'macro_P': 0.47638577912931257, 'macro_R': 0.612403437175092, 'macro_F1': 0.5085112827611473}


## Confronto finale tra:
- baseline legal-only
- best global tuning
- predicate-specific tuning

In [21]:
rows = []

for name, pred_map in [
    ("baseline_legal_only", pred_by_doc_base),
    ("best_global", pred_by_doc_best),
    ("predicate_specific", pred_by_doc_ps),
]:
    mi = micro_scores(gold_by_doc, pred_map)
    ma = macro_scores(gold_by_doc, pred_map)
    rows.append({
        "name": name,
        "micro_P": mi["P"],
        "micro_R": mi["R"],
        "micro_F1": mi["F1"],
        "macro_P": ma["macro_P"],
        "macro_R": ma["macro_R"],
        "macro_F1": ma["macro_F1"],
        "TP": mi["TP"],
        "FP": mi["FP"],
        "FN": mi["FN"],
    })

for r in rows:
    print(r)

{'name': 'baseline_legal_only', 'micro_P': 0.48172323759791125, 'micro_R': 0.6479367866549605, 'micro_F1': 0.5526020217147136, 'macro_P': 0.4583215878976895, 'macro_R': 0.6128277250936436, 'macro_F1': 0.4929603270235719, 'TP': 738, 'FP': 794, 'FN': 401}
{'name': 'best_global', 'micro_P': 0.49793672627235214, 'micro_R': 0.6356453028972783, 'micro_F1': 0.5584265329733898, 'macro_P': 0.47448121377417574, 'macro_R': 0.5934317700932226, 'macro_F1': 0.49312545875448127, 'TP': 724, 'FP': 730, 'FN': 415}
{'name': 'predicate_specific', 'micro_P': 0.5048076923076923, 'micro_R': 0.6453028972783144, 'micro_F1': 0.5664739884393064, 'macro_P': 0.47638577912931257, 'macro_R': 0.612403437175092, 'macro_F1': 0.5085112827611473, 'TP': 735, 'FP': 721, 'FN': 404}


## Export dei JSON migliori

Salviamo sia:
- la miglior configurazione globale
- la configurazione predicate-specific

Così puoi confrontarle direttamente nei submission file.

In [22]:
os.makedirs("predictions", exist_ok=True)

out_global = os.path.join(
    "predictions",
    f"bert_re_inference_legal_best_chars{BEST_MAX}_minp{BEST_MINP}_m{BEST_MARG}_T{BEST_TEMP}_renorm{int(BEST_RENORM)}.json"
)
with open(out_global, "w", encoding="utf-8") as f:
    json.dump(predictions_best, f, ensure_ascii=False, indent=2)

out_ps = os.path.join("predictions", "bert_re_inference_predicate_specific.json")
with open(out_ps, "w", encoding="utf-8") as f:
    json.dump(predictions_ps, f, ensure_ascii=False, indent=2)

print("saved:", out_global)
print("saved:", out_ps)

saved: predictions\bert_re_inference_legal_best_chars250_minp0.1_m0.25_T1.25_renorm1.json
saved: predictions\bert_re_inference_predicate_specific.json
